# SigAlg's `corr` method

In [50]:
# If running in Google Colab, uncomment the line below and run this cell first
# !pip install sigalg

The `Operators.corr` method in SigAlg is a method for computing *correlations* of random variables, both unconditional and conditional versions. The API reference is [here](https://johnmyers-phd.com/sigalg/api/core/#sigalg.core.Operators.corr).

## Mathematical definition

Let $X,Y:\Omega \to \mathbb{R}$ be two random variables on a probability space $(\Omega, \mathcal{F},P)$ for which $E(X^2), E(Y^2) < \infty$, and let $\mathcal{G}$ be a sub-$\sigma$-algebra of $\mathcal{F}$. The *conditional correlation* of $X$ and $Y$ with respect to $\mathcal{G}$ is any $\mathcal{G}$-measurable random variable $\rho(X, Y \mid \mathcal{G})$ for which

$$
\rho(X,Y\mid \mathcal{G}) = \frac{\sigma(X,Y \mid \mathcal{G})}{\sigma(X\mid \mathcal{G})\sigma(Y\mid \mathcal{G})},
$$

provided that the standard deviations in the denominator are nonzero. In the case that $\Omega$ is finite (as it always is, in SigAlg), the $\sigma$-algebra $\mathcal{G}$ is determined by its (finitely many) atoms, and the space $L^2(\Omega, \mathcal{G}, P)$ has an orthogonal basis given by the indicator functions of the atoms of $\mathcal{G}$ with nonzero probability. Then we have

$$
\rho(X,Y\mid \mathcal{G}) = \sum_B \rho(X|_B, Y|_B) I_B,
$$

where the sum extends over all atoms $B$ of $\mathcal{G}$ with nonzero probability, and where $\rho(X|_B, Y|_B)$ is the correlation of the restricted random variables $X|_B, Y|_B:B\to \mathbb{R}$ where $B$ is equipped with the conditional probability measure $P_B$ such that $P_B(C) = P(C)/P(B)$ for $C\subset B$.

## API examples


### Unconditional correlations

We begin by defining a sample space $\Omega = \{0,1,2,3,4\}$ and a probability measure $P$ on $\Omega$.

In [51]:
import numpy as np

from sigalg.core import ProbabilityMeasure, SampleSpace

rng = np.random.default_rng(42)

Omega = SampleSpace().from_sequence(size=5)
P = ProbabilityMeasure(sample_space=Omega).from_rand(random_state=rng)
print(P)

Probability measure 'P':
        probability
sample             
0          0.320930
1          0.311850
2          0.318334
3          0.037349
4          0.011538


Define two random variables $X,Y: \Omega \to \mathbb{R}$ on the sample space $\Omega$ and set their `probability_measure` attribute to $P$ so that all correlations will be computed relative to $P$.

In [52]:
from sigalg.core import RandomVariable

X = RandomVariable(domain=Omega).from_randint(low=-20, high=21, random_state=rng)
Y = RandomVariable(domain=Omega, name="Y").from_randint(
    low=-10, high=11, random_state=rng
)
X.probability_measure = P
Y.probability_measure = P

print(X, "\n")
print(Y)

Random variable 'X':
         X
sample    
0        1
1       20
2       10
3       11
4        9 

Random variable 'Y':
        Y
sample   
0       6
1       0
2      -8
3       7
4      -1


All correlations in SigAlg are instances of `RandomVariable`. The unconditional correlation `corr(X, Y)` is thus a constant random variable whose value is the usual correlation $\rho(X,Y) = \frac{\sigma(X,Y)}{\sigma(X)\sigma(Y)}$. The `item` method extracts $\rho(X,Y)$ from `corr(X, Y)`.

In [53]:
from sigalg.core import Operators

corr = Operators.corr

corr_rv = corr(X, Y)
corr_item = corr(X, Y).item()

print(corr_rv, "\n")
print(corr_item)

Random variable 'corr(X, Y)':
        corr(X, Y)
sample            
0        -0.386861
1        -0.386861
2        -0.386861
3        -0.386861
4        -0.386861 

-0.3868608444026747


### Conditional correlations

Define a $\sigma$-algebra $\mathcal{G}$ on $\Omega$ with atoms $A_0=\{0,1\}$ and $A_1 = \{2,3,4\}$. 

In [54]:
from sigalg.core import SigmaAlgebra

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 1,
    }
)

Compute the conditional correlation $\rho(X,Y\mid \mathcal{G})$.

In [55]:
print(corr(X, Y, G))

Random variable 'corr(X, Y|G)':
        corr(X, Y|G)
sample              
0           -1.00000
1           -1.00000
2            0.71463
3            0.71463
4            0.71463


### Testing properties of correlations

#### Conditional correlations are linear combinations

We noted in the definition that the conditional correlation may be expressed as a linear combination of the indicator functions of the atoms of the $\sigma$-algebra. In the next code cell, we test this. Notice that the output matches the output above.

In [56]:
I = RandomVariable.indicator_of

linear_combo = sum([corr(X(A), Y(A)).item() * I(A) for A in G.to_atoms()])

print(linear_combo.with_name("linear_combo"))

Random variable 'linear_combo':
        linear_combo
sample              
0           -1.00000
1           -1.00000
2            0.71463
3            0.71463
4            0.71463


### Perfectly correlated random variables

The correlation of two random variables $X$ and $Y$ that are perfectly correlated is $-1$ or $1$. In the next code cell, we test the conditional version of this:

In [57]:
Omega = SampleSpace().from_sequence(size=4)

X = RandomVariable(domain=Omega).from_dict(
    {
        0: -1,  # on the line y = x
        1: 1,  # on the line y = x
        2: -1,  # on the line y = -x
        3: 1,  # on the line y = -x
    }
)

Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: -1,  # on the line y = x
        1: 1,  # on the line y = x
        2: 1,  # on the line y = -x
        3: -1,  # on the line y = -x
    }
)

P = ProbabilityMeasure(sample_space=Omega).from_rand(random_state=rng)
X.probability_measure = P
Y.probability_measure = P

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
    }
)

print(corr(X, Y, G))

Random variable 'corr(X, Y|G)':
        corr(X, Y|G)
sample              
0                1.0
1                1.0
2               -1.0
3               -1.0


#### Independent implies uncorrelated

If $X$ and $Y$ are two independent random variables, then $X$ and $Y$ are uncorrelated, i.e., $\rho(X,Y)=0$. In the next code cell, we test this.

In [58]:
from scipy.stats import bernoulli

from sigalg.core import Time
from sigalg.processes import IIDProcess

coin_flip = IIDProcess(
    distribution=bernoulli(p=0.7),
    support=[0, 1],
    time=Time.discrete(length=1),
    name="coin_flip",
).from_enumeration()

X, Y = coin_flip
P = coin_flip.probability_measure
X.with_name("X")
Y.with_name("Y")

print(corr(X, Y))

Random variable 'corr(X, Y)':
              corr(X, Y)
trajectory              
0          -2.643388e-16
1          -2.643388e-16
2          -2.643388e-16
3          -2.643388e-16
